# 📈 SICAV Fund Comparator — Google Colab

Questo notebook installa le dipendenze, carica i moduli e avvia la pipeline di analisi.

**Struttura:**
1. Installazione dipendenze
2. Upload dei file `analysis.py` e `app.py`
3. Esecuzione del modulo di analisi
4. Avvio della Streamlit app tramite tunnel `localtunnel`


In [ ]:
# ============================================================
# CELLA 1 — Installazione dipendenze
# Esegui questa cella per prima cosa.
# ============================================================
!pip install pandas numpy yfinance openpyxl streamlit matplotlib --quiet
print('✅ Dipendenze installate correttamente.')

In [ ]:
# ============================================================
# CELLA 2 — Upload file sorgente
# Carica analysis.py e app.py nella directory /content di Colab.
# Opzione A: usa il pannello file a sinistra per il drag & drop.
# Opzione B: esegui questa cella per upload interattivo.
# ============================================================
from google.colab import files

print('Carica analysis.py:')
uploaded = files.upload()  # Seleziona analysis.py

print('Carica app.py:')
uploaded = files.upload()  # Seleziona app.py

import os
print('File in /content:', os.listdir('/content'))

In [ ]:
# ============================================================
# CELLA 3 — Test modulo analysis.py
# Scarica i dati e stampa il summary. Non richiede Streamlit.
# ============================================================
import sys
sys.path.insert(0, '/content')  # Assicura che Python trovi i file caricati

from analysis import run_analysis

df_nav, df_cum, df_summary = run_analysis(
    start_date='2021-01-01',
    end_date='2024-12-31',
    comparison_freq='W-FRI',
    max_gap_days=20,
)

print('\n--- SUMMARY ---')
print(df_summary)

print('\n--- CUMULATIVE (prime 5 righe) ---')
print(df_cum.head())

In [ ]:
# ============================================================
# CELLA 4 — Export Excel (opzionale)
# Genera il file Excel e lo scarica sul tuo computer.
# ============================================================
from analysis import export_excel
from google.colab import files

output_path = '/content/sicav_comparison_output.xlsx'
export_excel(df_nav, df_cum, df_summary, output_path)

files.download(output_path)
print('✅ File Excel scaricato.')

In [ ]:
# ============================================================
# CELLA 5 — Avvio Streamlit app tramite localtunnel
#
# ISTRUZIONI:
# 1. Esegui questa cella.
# 2. Attendi che appaia il messaggio 'your url is: https://...'.
# 3. Clicca sul link per aprire la app nel browser.
# 4. Se richiesto, inserisci l'IP mostrato nel campo 'Endpoint IP'.
#    L'IP è quello stampato sotto ('External IP').
#
# NOTA: localtunnel è la soluzione più semplice e gratuita.
#       In alternativa puoi usare ngrok (richiede account e token).
# ============================================================

# Installa localtunnel
!npm install -q localtunnel

# Mostra l'IP esterno del runtime Colab (serve per autenticarsi su localtunnel)
import urllib.request
external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f'External IP (da usare come Endpoint IP su localtunnel): {external_ip}')

# Avvia Streamlit in background sulla porta 8501
import subprocess
proc = subprocess.Popen(
    ['streamlit', 'run', '/content/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

import time
time.sleep(5)  # Attendi avvio Streamlit

# Avvia il tunnel localtunnel sulla porta 8501
!npx localtunnel --port 8501

In [ ]:
# ============================================================
# CELLA 6 — Alternativa: ngrok (se localtunnel non funziona)
#
# Richiede account gratuito su https://ngrok.com
# e il tuo authtoken personale.
#
# Sostituisci YOUR_NGROK_TOKEN con il tuo token.
# ============================================================

# !pip install pyngrok -q
# from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')  # <-- sostituisci con il tuo token
#
# import subprocess, time
# proc = subprocess.Popen(
#     ['streamlit', 'run', '/content/app.py',
#      '--server.port', '8501', '--server.headless', 'true'],
#     stdout=subprocess.PIPE, stderr=subprocess.PIPE
# )
# time.sleep(5)
#
# public_url = ngrok.connect(8501)
# print(f'Streamlit app disponibile su: {public_url}')